# IoU metric for one image

This notebook measures how close the mask predicted by the model is to the reference mask, for a single image, with the Intersection over Union (IoU).

For a class, the IoU is the number of pixels labelled with this class in both masks (the intersection), divided by the number of pixels labelled with it in at least one of the two masks (the union). It goes from 0 (no overlap) to 1 (identical masks).

The predicted mask is the one saved by the notebook `02_segmentation_api.ipynb` in `data/output/`, so no API call is made here.

In [ ]:
%load_ext autoreload
%autoreload 2

## Configuration

* `IMAGE_NUMBER`: the image to evaluate. Its predicted mask must exist in `data/output/`, run `02_segmentation_api.ipynb` or `scripts/segment_images.py` first.

In [ ]:
import re
import warnings
from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt
from matplotlib.patches import Patch
from PIL import Image

from oc_p02_fashion_segmentation.constants import (
    ACCESSORY_CLASS_IDS,
    CLOTHING_CLASS_IDS,
    ID_TO_CLASS,
    SHOE_CLASS_IDS,
)
from oc_p02_fashion_segmentation.masks import mask_path_for
from oc_p02_fashion_segmentation.metrics import (
    intersection_and_union,
    iou_by_class,
    iou_from_counts,
)
from oc_p02_fashion_segmentation.visualization import (
    CLASS_COLORS,
    display_comparison,
    draw_overlay,
)

DATA_DIR = Path("../data/input")
OUTPUT_DIR = Path("../data/output")
IMAGE_NUMBER = 3
IMAGE_PATH = DATA_DIR / "IMG" / f"image_{IMAGE_NUMBER}.png"
REFERENCE_MASK_PATH = DATA_DIR / "Mask" / f"mask_{IMAGE_NUMBER}.png"
PREDICTED_MASK_PATH = mask_path_for(IMAGE_PATH, OUTPUT_DIR)

## Load the masks

Both masks are single-channel images where each pixel holds a class ID.

In [ ]:
image_array = np.array(Image.open(IMAGE_PATH))
reference_mask = np.array(Image.open(REFERENCE_MASK_PATH))
predicted_mask = np.array(Image.open(PREDICTED_MASK_PATH))

print(f"Reference mask: {reference_mask.shape}, predicted mask: {predicted_mask.shape}")

## Compute the IoU by class

`iou_by_class` returns one value per class ID. A class absent from both masks has an undefined IoU (an empty union) and gets `nan`, so it is left out of the mean instead of counting as a perfect or a null score.

In [ ]:
iou = iou_by_class(reference_mask, predicted_mask)

reference_pixels = np.bincount(reference_mask.ravel(), minlength=len(iou))
predicted_pixels = np.bincount(predicted_mask.ravel(), minlength=len(iou))

print(f"{'class':<15}{'reference px':>14}{'predicted px':>14}{'IoU':>8}")
for class_id, value in enumerate(iou):
    iou_text = "n/a" if np.isnan(value) else f"{value:.3f}"
    print(
        f"{ID_TO_CLASS[class_id]:<15}{reference_pixels[class_id]:>14}"
        f"{predicted_pixels[class_id]:>14}{iou_text:>8}"
    )

## Mean IoU

The mean IoU (mIoU) averages the IoU of the classes present in at least one of the two masks, with `np.nanmean`. It is given with and without the background, which is large and easy to predict, so it raises the score.

In [ ]:
print(f"mIoU, all classes:        {np.nanmean(iou):.3f}")
print(f"mIoU, without background: {np.nanmean(iou[1:]):.3f}")

## Compare visually

The IoU can be checked against the pixels where the two masks disagree: the classes with the lowest IoU are the ones with the most disagreement.

In [ ]:
display_comparison(image_array, reference_mask, predicted_mask)

## IoU on the whole dataset

The IoU of a dataset can be aggregated in two ways, which give different results:

* **mean of the IoU by image**: compute the IoU of each class for each image, then average over the images. Every image weighs the same, whatever the size of its objects. An image where the class is absent from both masks has no IoU and is left out of the average.
* **IoU over all the images**: sum the intersection and the union pixel counts of each class over all the images, then compute one IoU. Every pixel weighs the same, so images with large objects count more.

Both start from the per-image counts given by `intersection_and_union`, computed here for the 50 images of the dataset.

In [ ]:
def image_number(path):
    return int(re.search(r"\d+", path.stem).group())


image_paths = sorted((DATA_DIR / "IMG").glob("image_*.png"), key=image_number)

intersections = []
unions = []
for path in image_paths:
    number = image_number(path)
    reference = np.array(Image.open(DATA_DIR / "Mask" / f"mask_{number}.png"))
    predicted = np.array(Image.open(OUTPUT_DIR / f"mask_{number}.png"))
    intersection, union = intersection_and_union(reference, predicted)
    intersections.append(intersection)
    unions.append(union)

# One row per image, one column per class.
intersections = np.array(intersections)
unions = np.array(unions)
print(f"{len(image_paths)} images, counts of shape {intersections.shape}")

### Method 1: mean of the IoU by image

`iou_from_counts` works on the whole `(images, classes)` arrays at once, and gives `nan` for each class missing from both masks of an image. `np.nanmean` then ignores them.

A class that appears in no image at all has no IoU: it stays `nan`.

In [ ]:
iou_by_image = iou_from_counts(intersections, unions)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)  # Class absent from all images.
    mean_iou_by_class = np.nanmean(iou_by_image, axis=0)
images_with_class = np.count_nonzero(unions > 0, axis=0)

### Method 2: IoU over all the images

The counts are summed over the images before computing the IoU.

In [ ]:
dataset_iou_by_class = iou_from_counts(intersections.sum(axis=0), unions.sum(axis=0))

### Comparison

The table gives, for each class, the number of images where it appears in at least one of the two masks, and the IoU with both methods.

In [ ]:
def format_iou(value):
    return "n/a" if np.isnan(value) else f"{value:.3f}"


print(f"{'class':<15}{'images':>8}{'mean by image':>15}{'over all images':>17}")
for class_id in range(len(dataset_iou_by_class)):
    print(
        f"{ID_TO_CLASS[class_id]:<15}{images_with_class[class_id]:>8}"
        f"{format_iou(mean_iou_by_class[class_id]):>15}"
        f"{format_iou(dataset_iou_by_class[class_id]):>17}"
    )

print()
print(f"{'mIoU':<23}{'mean by image':>15}{'over all images':>17}")
print(
    f"{'all classes':<23}{np.nanmean(mean_iou_by_class):>15.3f}"
    f"{np.nanmean(dataset_iou_by_class):>17.3f}"
)
print(
    f"{'without background':<23}{np.nanmean(mean_iou_by_class[1:]):>15.3f}"
    f"{np.nanmean(dataset_iou_by_class[1:]):>17.3f}"
)

## IoU by group of classes

The classes are also gathered in groups, defined in `constants.py`. The two methods above are used for each group, and the number of images where the group appears in at least one of the two masks is given.

The groups do not all work the same way:

* **clothing** (upper-clothes, skirt, pants, dress) and **accessories** (hat, sunglasses, belt, bag, scarf): the classes stay distinct. The IoU of the group is the mean of the IoU of its classes, so confusing a skirt with a dress is still an error. A group appears in an image if any of its classes does.
* **shoes** (left and right shoe): the classes are merged into a single class, before the IoU is computed. Confusing the left shoe with the right one is not an error, only missing a shoe or predicting one where there is none is.

In [ ]:
def format_row(name, images, mean_by_image, over_all_images):
    print(
        f"{name:<15}{images:>8}"
        f"{format_iou(mean_by_image):>15}{format_iou(over_all_images):>17}"
    )


print(f"{'group':<15}{'images':>8}{'mean by image':>15}{'over all images':>17}")
for name, class_ids in [
    ("Clothing", CLOTHING_CLASS_IDS),
    ("Accessories", ACCESSORY_CLASS_IDS),
]:
    format_row(
        name,
        np.count_nonzero((unions[:, class_ids] > 0).any(axis=1)),
        np.nanmean(mean_iou_by_class[class_ids]),
        np.nanmean(dataset_iou_by_class[class_ids]),
    )

### Shoes: left and right merged

Each mask is rewritten with the ID of the left shoe on every shoe pixel, then the counts and both aggregations are computed again on this 17-class labelling, as for the other classes.

In [ ]:
MERGED_SHOE_ID = SHOE_CLASS_IDS[0]


def merge_shoes(mask):
    return np.where(np.isin(mask, SHOE_CLASS_IDS), MERGED_SHOE_ID, mask)


shoe_intersections = []
shoe_unions = []
for path in image_paths:
    number = image_number(path)
    reference = np.array(Image.open(DATA_DIR / "Mask" / f"mask_{number}.png"))
    predicted = np.array(Image.open(OUTPUT_DIR / f"mask_{number}.png"))
    intersection, union = intersection_and_union(
        merge_shoes(reference), merge_shoes(predicted)
    )
    shoe_intersections.append(intersection[MERGED_SHOE_ID])
    shoe_unions.append(union[MERGED_SHOE_ID])

shoe_intersections = np.array(shoe_intersections)
shoe_unions = np.array(shoe_unions)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)  # Shoes absent from all images.
    shoe_mean_by_image = np.nanmean(iou_from_counts(shoe_intersections, shoe_unions))
shoe_over_all_images = iou_from_counts(shoe_intersections.sum(), shoe_unions.sum())

print(f"{'group':<15}{'images':>8}{'mean by image':>15}{'over all images':>17}")
format_row(
    "Shoes (merged)",
    np.count_nonzero(shoe_unions > 0),
    shoe_mean_by_image,
    shoe_over_all_images,
)

## Sensitivity analysis: without the inaccurate reference masks

In `01_data_exploration.ipynb`, three images were spotted with an inaccurate reference mask: 1, 26 and 29. For these images, a low IoU can come from the reference as much as from the model.

To see if they change the conclusions, all the scores above are computed again on the 47 other images, and compared with the 50 images. The difference is `filtered - full`: a positive value means the score is higher without these images.

In [ ]:
EXCLUDED_IMAGES = [1, 26, 29]

numbers = np.array([image_number(path) for path in image_paths])
kept = ~np.isin(numbers, EXCLUDED_IMAGES)
print(f"{np.count_nonzero(kept)} images kept out of {len(numbers)}")

In [ ]:
def aggregate(selected):
    """Return {name: (mean of the IoU by image, IoU over all images)}."""
    selected_intersections = intersections[selected]
    selected_unions = unions[selected]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)  # Class absent from all images.
        mean_by_image = np.nanmean(
            iou_from_counts(selected_intersections, selected_unions), axis=0
        )
        shoe_mean_by_image = np.nanmean(
            iou_from_counts(shoe_intersections[selected], shoe_unions[selected])
        )
    over_all_images = iou_from_counts(
        selected_intersections.sum(axis=0), selected_unions.sum(axis=0)
    )
    shoe_over_all_images = iou_from_counts(
        shoe_intersections[selected].sum(), shoe_unions[selected].sum()
    )

    scores = {
        ID_TO_CLASS[class_id]: (mean_by_image[class_id], over_all_images[class_id])
        for class_id in range(len(mean_by_image))
    }
    for name, class_ids in [
        ("Clothing", CLOTHING_CLASS_IDS),
        ("Accessories", ACCESSORY_CLASS_IDS),
    ]:
        scores[name] = (
            np.nanmean(mean_by_image[class_ids]),
            np.nanmean(over_all_images[class_ids]),
        )
    scores["Shoes (merged)"] = (shoe_mean_by_image, shoe_over_all_images)
    scores["mIoU, all classes"] = (np.nanmean(mean_by_image), np.nanmean(over_all_images))
    scores["mIoU, no background"] = (
        np.nanmean(mean_by_image[1:]),
        np.nanmean(over_all_images[1:]),
    )
    return scores


full_scores = aggregate(np.ones(len(numbers), dtype=bool))
filtered_scores = aggregate(kept)

In [ ]:
def format_difference(value):
    return "n/a" if np.isnan(value) else f"{value:+.3f}"


print(f"{'':<21}{'mean of the IoU by image':^30}{'IoU over all images':^30}")
print(f"{'':<21}" + 2 * f"{'50':>10}{'47':>10}{'diff':>10}")
for name in full_scores:
    if name == "Clothing" or name == "mIoU, all classes":
        print()
    cells = []
    for method in (0, 1):
        full = full_scores[name][method]
        filtered = filtered_scores[name][method]
        cells.append(
            f"{format_iou(full):>10}{format_iou(filtered):>10}"
            f"{format_difference(filtered - full):>10}"
        )
    print(f"{name:<21}" + "".join(cells))

### Conclusion

Removing images 1, 26 and 29 changes the overall scores very little: the mIoU moves by +0.017 with the mean of the IoU by image, and by +0.003 with the IoU over all images. The second method is the more stable, as it weighs pixels and these images hold few of them.

The differences concentrate on a few classes:

* **dress** (+0.054 and +0.027), **pants** (+0.025 and +0.017) and **scarf** (+0.228 and +0.032): the model is penalised on these images, which is consistent with an inaccurate reference.
* the scarf jump with the mean by image comes from the 3 images where it appears: removing one of them moves the average a lot. It is a small sample effect, not a reliable gain.
* the shoes, once merged, and most other classes move by less than 0.01.

So the three inaccurate references are not a major source of error: they slightly lower the scores of a few classes, but do not change the ranking of the classes, nor the conclusions of the evaluation.

Isolated annotation errors weigh little compared to a systematic deviation from the convention: the three inaccurate references move the scores by a few hundredths at most, while the left and right shoes, swapped systematically between the reference and the model, cost about 0.4 of IoU (0.19 to 0.31 as separate classes, against 0.73 once merged).


## Images ranked by IoU of the clothes

The 50 images are sorted from the best to the worst clothes score: the mean of the IoU of the clothing classes (upper-clothes, skirt, pants, dress), using the per-image IoU computed above. A clothing class absent from both masks of an image is left out of its mean.

Each image is shown with the predicted mask overlaid. Under it, the legend gives the IoU of every class present in the reference or the predicted mask. The clothing classes, the ones used for the ranking, are in bold.


In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)  # No clothing in an image.
    clothes_miou = np.nanmean(iou_by_image[:, CLOTHING_CLASS_IDS], axis=1)

# nan (no clothing at all) is sorted last.
ranking = np.argsort(-clothes_miou)

COLUMNS = 5
ROWS_PER_FIGURE = 2
IMAGE_HEIGHT = 5.5  # inches
# The legend has its own axes, tall enough for the image with the most classes,
# so it never overlaps the row below.
MAX_LEGEND_ENTRIES = np.count_nonzero(~np.isnan(iou_by_image), axis=1).max()
LEGEND_HEIGHT = 0.25 * MAX_LEGEND_ENTRIES  # inches, for a font size of 8

for start in range(0, len(ranking), COLUMNS * ROWS_PER_FIGURE):
    chunk = ranking[start : start + COLUMNS * ROWS_PER_FIGURE]
    fig = plt.figure(
        figsize=(3.6 * COLUMNS, (IMAGE_HEIGHT + LEGEND_HEIGHT) * ROWS_PER_FIGURE)
    )
    grid = fig.add_gridspec(
        2 * ROWS_PER_FIGURE,
        COLUMNS,
        height_ratios=[IMAGE_HEIGHT, LEGEND_HEIGHT] * ROWS_PER_FIGURE,
        hspace=0.05,
    )

    for position, index in enumerate(chunk):
        row, column = divmod(position, COLUMNS)
        ax_image = fig.add_subplot(grid[2 * row, column])
        ax_legend = fig.add_subplot(grid[2 * row + 1, column])
        ax_legend.axis("off")

        number = numbers[index]
        draw_overlay(
            ax_image,
            np.array(Image.open(image_paths[index])),
            np.array(Image.open(OUTPUT_DIR / f"mask_{number}.png")),
            alpha=0.6,
        )
        ax_image.set_title(
            f"#{start + position + 1} image {number}\n"
            f"clothes mIoU {format_iou(clothes_miou[index])}"
        )

        present = np.flatnonzero(~np.isnan(iou_by_image[index]))
        legend = ax_legend.legend(
            handles=[
                Patch(
                    color=CLASS_COLORS[ID_TO_CLASS[class_id]],
                    label=f"{ID_TO_CLASS[class_id]}: {iou_by_image[index, class_id]:.3f}",
                )
                for class_id in present
            ],
            loc="upper center",
            frameon=False,
            fontsize=8,
        )
        for class_id, text in zip(present, legend.get_texts()):
            if class_id in CLOTHING_CLASS_IDS:
                text.set_fontweight("bold")

    plt.show()

### Observations and conclusion

**Top of the ranking.** The best images (clothes mIoU from 0.95 to 0.97) look alike: full-length photos, taken from the front, of a person standing out from the background and wearing a clearly distinct outfit.

**Bottom of the ranking.** The five worst images (clothes mIoU from 0.77 down to 0.30) are not badly segmented everywhere: they come from a few causes, and most of them are not boundary errors.

* **image 29 (0.30)** This is one of the three inaccurate references spotted in `01_data_exploration.ipynb`. The reference says “dress,” while the model says “top + shorts.” The reference is incorrect, where the model correctly categorized them.
* **image 31 (0.30)** a long top that the reference calls a dress and the model categorized as upper-clothes. This isn’t really an error, it’s a category ambiguity.
* **image 11 (0.59)** the same issue between shorts and a skirt, compounded by a watermark on the image.
* **image 48 (0.70)** shorts almost entirely hidden under a long shirt. The small, partially obscured object results in a low IoU.
* **image 9 (0.77)** the image is 97 pixels wide. It’s the low resolution that’s holding the model back.

**Conclusion.** The model segments the clothes very well when the outfit is conventional: a clear top and bottom, or a plain dress, in front of a background. Its low scores come mostly from confusion between similar categories (dresses or long tops, shorts or skirts), incorrect reference and poor-quality images.